# TikTok-10M on Colab: ingest → splits → leaderboard

Runs the CPU part of the pipeline on Colab's disk instead of an external drive. No GPU needed (choose a CPU runtime; high-RAM helps but isn't required).

1. Downloads `The-data-company/TikTok-10M` (~10 GB) to the Colab disk.
2. Streams it into one deduped post table plus a data card (`scale.ingest_tiktok10m`).
3. Builds the post table with Emily's CSV (`scale.build_posts`), writes the shared splits (`scale.splits`) and scores the built-in baselines (`scale.leaderboard`).
4. Copies the outputs (a few hundred MB) to Google Drive. The raw shards stay on the Colab disk and disappear with the session.

Set `DRY_RUN = True` first to check everything on synthetic shards in about a minute.

In [ ]:
BRANCH = "claude/project-thread-e8hh2j"   # switch to master once PR #5 is merged
DRY_RUN = False        # True: synthetic shards, no download
MAX_SHARDS = None      # e.g. 1 for a quick first look at one real shard
TEST_FRAC = 0.1        # raise (e.g. 0.2) if the strict test split comes out small
TAG = "dryrun" if DRY_RUN else "full"   # dry and real runs never share folders
SRC = f"/content/hf/tiktok10m_{TAG}"
OUT = f"/content/scale_{TAG}"
DRIVE_OUT = f"/content/drive/MyDrive/tiktok/scale_{TAG}"   # where results are kept

In [ ]:
%cd /content
!rm -rf /content/repo
!git clone -q --branch $BRANCH https://github.com/jason-d-garcia/tiktok-virality-prediction.git /content/repo
%cd /content/repo
!pip -q install pyarrow huggingface_hub
# pandas/numpy/scipy from Colab's image are enough for these steps; torch is only needed for training

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os; os.makedirs(DRIVE_OUT, exist_ok=True)

## Download TikTok-10M

If the download asks for authentication, run `from huggingface_hub import notebook_login; notebook_login()` with a read token first.

In [ ]:
if DRY_RUN:
    !python -m tests.synthetic_tiktok10m --out $SRC --shards 3 --creators 400
else:
    from huggingface_hub import snapshot_download
    snapshot_download("The-data-company/TikTok-10M", repo_type="dataset", local_dir=SRC,
                      allow_patterns=["*.parquet"], max_workers=8)
!du -sh $SRC; find $SRC -name "*.parquet" | head -12

## Ingest (streams one row group at a time; re-running skips finished shards)

In [ ]:
shards_arg = f"--max-shards {MAX_SHARDS}" if MAX_SHARDS else ""
!python -m scale.ingest_tiktok10m --src $SRC --out $OUT/tiktok10m_posts.parquet $shards_arg
import json; card = json.load(open(f"{OUT}/tiktok10m_posts.card.json"))
{k: card[k] for k in ["raw_rows", "unique_posts", "duplicate_rows", "creators", "share_below_truncation",
                      "log10_views_hist", "age_days_quantiles"]}

## Post table, splits, baseline leaderboard

`build_posts` caps TikTok-10M at whole creators (`--max-posts-per-source`); 200K posts keeps every later step fast. Raise it once the pipeline is proven.

In [ ]:
# build_posts samples whole creators from narrow columns before loading rows, so the
# 10M-row table never sits in memory. Check this cell's output for errors before moving on.
!python -m scale.build_posts --emily data/raw/tiktok_stats.csv --tiktok10m $OUT/tiktok10m_posts.parquet \
    --max-posts-per-source 200000 --out $OUT/posts.parquet
!python -m scale.splits --posts $OUT/posts.parquet --out $OUT/splits.parquet --test-frac $TEST_FRAC
for scheme in ["strict", "creator", "time"]:
    !python -m scale.leaderboard --posts $OUT/posts.parquet --splits $OUT/splits.parquet --scheme $scheme --out $OUT/leaderboard

In [ ]:
from IPython.display import Markdown
Markdown(open(f"{OUT}/leaderboard_strict.md").read())

## Keep the results on Drive

In [ ]:
!cp $OUT/*.parquet $OUT/*.json $OUT/leaderboard_* $DRIVE_OUT/
!ls -lh $DRIVE_OUT